# NuNER Dataset Preprocessing

Load and preprocess the [NuNER](https://huggingface.co/datasets/numind/NuNER) dataset for information extraction evaluation.

In [ ]:
import ast
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset, DatasetDict

## Load Dataset

In [ ]:
ds = load_dataset("numind/NuNER", split="full")
print(f"Loaded {len(ds):,} rows")
print(f"Columns: {ds.column_names}")

## Basic EDA

In [ ]:
df = ds.to_pandas()
print(f"Shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum():,}")
df.head()

## Parse Output Column

Convert `"['entity <> type', ...]"` strings into structured lists of `{"entity": ..., "type": ...}` dicts.

In [ ]:
def parse_entities(output_str: str) -> list[dict]:
    """Parse an output string into a list of {entity, type} dicts."""
    try:
        items = ast.literal_eval(output_str)
    except (ValueError, SyntaxError):
        return []
    entities = []
    for item in items:
        parts = item.split(" <> ", maxsplit=1)
        if len(parts) == 2:
            entities.append({"entity": parts[0].strip(), "type": parts[1].strip()})
    return entities


# Test on a few rows
for i in range(3):
    print(f"Input:  {df.iloc[i]['input'][:100]}")
    print(f"Parsed: {parse_entities(df.iloc[i]['output'])}")
    print()

## Entity Type Analysis

In [ ]:
type_counter = Counter()
parse_failures = 0

for output_str in df["output"]:
    parsed = parse_entities(output_str)
    if not parsed and output_str != "[]":
        parse_failures += 1
    for ent in parsed:
        type_counter[ent["type"]] += 1

print(f"Unique entity types: {len(type_counter)}")
print(f"Parse failures: {parse_failures:,}")
print(f"Total entities: {sum(type_counter.values()):,}")
print(f"\nTop 20 types:")
for etype, count in type_counter.most_common(20):
    print(f"  {etype:30s} {count:>10,}")

In [ ]:
top20 = type_counter.most_common(20)
labels, counts = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(labels)), counts)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.set_title("Top 20 Entity Types in NuNER")
plt.tight_layout()
plt.show()

## Train / Val / Test Split (80/10/10)

In [ ]:
SEED = 42

# First split: 80% train, 20% temp
split1 = ds.train_test_split(test_size=0.2, seed=SEED)
# Second split: 50/50 on the 20% temp -> 10% val, 10% test
split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)

splits = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

for name, split in splits.items():
    print(f"{name:12s}: {len(split):>10,} rows")

## Save to Disk

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

for name, split in splits.items():
    out_path = data_dir / f"{name}.parquet"
    split.to_parquet(str(out_path))
    print(f"Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

print("\nDone!")